# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Show the dataset metadata (as attributes, not by subscripting)
print(f"Name: {dataset.metadata.name}")
print(f"Version: {dataset.metadata.version}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant schema organizes tabular data into **record sets**. Each record set has its own `@id`. We'll enumerate the available record sets and show their fields.


In [ ]:
from pprint import pprint

# List all record sets by @id and name
record_sets = dataset.metadata.record_set
print('Available Record sets:')
for rs in record_sets:
    print(f"  @id: {rs['@id']}, name: {rs.get('name', '<no name>')}")

# Display fields for each record set
for rs in record_sets:
    print(f"\nRecordSet @id: {rs['@id']}, name: {rs.get('name', '<no name>')}")
    fields = rs.get('field', [])
    if not isinstance(fields, list):
        fields = [fields]
    for f in fields:
        # f is a dict with ('@id', 'name', 'dataType')
        print(f"    Field @id: {f['@id']}, name: {f.get('name', '<no name>')}, dataType: {f.get('dataType', '<unknown>')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Entities are referenced by their `@id` fields.

You should choose a record set with meaningful tabular data. Here, we'll load the main data record set.

In [ ]:
# Choose which record set to use (set variable to its '@id')
# For this dataset, the main data usually has an @id like 'cr:RecordSet:..., but list them in previous cell.
# (Replace the below with your actual main record set @id)
main_record_set_id = dataset.metadata.record_set[0]['@id']

# You may add more record set @ids in this list if needed
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_set]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records from RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(df.head(2), '\n')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. We'll demonstrate with one numeric field from the main record set (by its `@id`).

In [ ]:
# Inspect which fields are numeric in the main record set
df = dataframes[main_record_set_id]
print('All columns:', df.columns.tolist())
print('\nData types:')
print(df.dtypes)

# Try to automatically pick a numeric field
numeric_fields = df.select_dtypes(include=['number']).columns.tolist()
if numeric_fields:
    numeric_field_id = numeric_fields[0]
    print(f"Using numeric field: {numeric_field_id}")
else:
    raise ValueError("No numeric fields detected in the main record set!")

threshold = df[numeric_field_id].mean() if df[numeric_field_id].count() > 0 else 0
print(f"Filtering rows where {numeric_field_id} > {threshold:.2f}")
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered: {len(filtered_df)} records")

# Normalize this numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try to group by a categorical field, if present
cat_fields = df.select_dtypes(include=['object']).columns.tolist()
group_field = None
if cat_fields:
    group_field = cat_fields[0]
    print(f"Grouping by: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. For example, let's visualize the distribution of the main numeric field, and if grouped data are available, plot the group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field_id].dropna(), kde=True)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If group_field/grouped_df is available, plot group means
if 'grouped_df' in locals():
    plt.figure(figsize=(10, 4))
    sns.barplot(data=grouped_df, x=group_field, y=numeric_field_id)
    plt.title(f'Mean {numeric_field_id} by {group_field}')
    plt.xlabel(group_field)
    plt.ylabel(f'Mean {numeric_field_id}')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
We have loaded, explored, and visualized the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using `mlcroissant`. This notebook demonstrated programmatic access to Croissant-structured data via `@id` references, showing how to extract, process, and analyze bioinformatics mass spectrometry datasets. Adapt the cell logic for deeper exploration according to your analysis goals!